# Importação de Bibliotecas

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Carregamento e divisão dos dados

In [2]:
url_titanic = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df_titanic = pd.read_csv(url_titanic)

X = df_titanic.drop(columns=['Survived'])
y = df_titanic['Survived']

num_features = ['Age', 'Fare']
cat_features = ['Sex', 'Embarked']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensões do Treino: {X_train.shape}")
print(f"Dimensões do Teste: {X_test.shape}")

Dimensões do Treino: (712, 11)
Dimensões do Teste: (179, 11)


# Criação e Avaliação de Pipelines

In [3]:
def executar_experimento(imputer_num, scaler_num, classificador, X_tr=X_train, X_te=X_test, y_tr=y_train, y_te=y_test):
    num_steps = []
    if imputer_num is not None:
        num_steps.append(('imputer', imputer_num))
    if scaler_num is not None:
        num_steps.append(('scaler', scaler_num))

    num_pipeline = Pipeline(num_steps) if num_steps else 'passthrough'

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    preprocessor = ColumnTransformer([
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)])

    model_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', classificador)])

    model_pipeline.fit(X_tr, y_tr)
    predicoes = model_pipeline.predict(X_te)

    return accuracy_score(y_te, predicoes)

# Execução dos experimentos

In [4]:
dicionario_resultados = {}

# --- Configuração Base (Referência) ---
dicionario_resultados['Pipeline Base (Mediana + StandardScaler + LogReg)'] = executar_experimento(
    imputer_num=SimpleImputer(strategy='median'),
    scaler_num=StandardScaler(),
    classificador=LogisticRegression(max_iter=1000, random_state=42)
)

# --- Mudança 1: Estratégia de Imputação (Média ou Constante) ---
dicionario_resultados['Imputação por Média'] = executar_experimento(
    imputer_num=SimpleImputer(strategy='mean'),
    scaler_num=StandardScaler(),
    classificador=LogisticRegression(max_iter=1000, random_state=42)
)

dicionario_resultados['Imputação por Constante (0)'] = executar_experimento(
    imputer_num=SimpleImputer(strategy='constant', fill_value=0),
    scaler_num=StandardScaler(),
    classificador=LogisticRegression(max_iter=1000, random_state=42)
)

# --- Mudança 2: Método de Escalonamento (MinMaxScaler) ---
dicionario_resultados['Escalonamento com MinMaxScaler'] = executar_experimento(
    imputer_num=SimpleImputer(strategy='median'),
    scaler_num=MinMaxScaler(),
    classificador=LogisticRegression(max_iter=1000, random_state=42)
)

# --- Mudança 3: Substituição do Modelo (Decision Tree) ---
dicionario_resultados['Modelo Decision Tree'] = executar_experimento(
    imputer_num=SimpleImputer(strategy='median'),
    scaler_num=None,
    classificador=DecisionTreeClassifier(random_state=42)
)

# --- Mudança 4: Ignorar Pré-processamento (Sem Escalonamento) ---
dicionario_resultados['Sem Escalonamento Numérico'] = executar_experimento(
    imputer_num=SimpleImputer(strategy='median'),
    scaler_num=None,
    classificador=LogisticRegression(max_iter=1000, random_state=42)
)

# --- Mudança 5: Ignorar Pré-processamento (Sem Tratamento de Nulos via Remoção) ---
df_limpo = df_titanic.dropna(subset=num_features + cat_features)
X_clean = df_limpo.drop(columns=['Survived'])
y_clean = df_limpo['Survived']

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

dicionario_resultados['Sem Imputação (Eliminação de Linhas)'] = executar_experimento(
    imputer_num=None,
    scaler_num=StandardScaler(),
    classificador=LogisticRegression(max_iter=1000, random_state=42),
    X_tr=X_tr_c, X_te=X_te_c, y_tr=y_tr_c, y_te=y_te_c
)

# Exibição de Resultados

In [5]:
df_resultados = pd.DataFrame(list(dicionario_resultados.items()), columns=['Cenário / Experimento', 'Acurácia obtida'])
df_resultados = df_resultados.sort_values(by='Acurácia obtida', ascending=False).reset_index(drop=True)

print("\n" + "="*60)
print("             TABELA COMPARATIVA DE RESULTADOS")
print("="*60)
print(df_resultados.to_string(index=False, formatters={'Acurácia obtida': '{:.4f}'.format}))
print("="*60 + "\n")


             TABELA COMPARATIVA DE RESULTADOS
                            Cenário / Experimento Acurácia obtida
                      Imputação por Constante (0)          0.7821
                   Escalonamento com MinMaxScaler          0.7821
Pipeline Base (Mediana + StandardScaler + LogReg)          0.7765
                              Imputação por Média          0.7765
                       Sem Escalonamento Numérico          0.7765
             Sem Imputação (Eliminação de Linhas)          0.7622
                             Modelo Decision Tree          0.7263



# Respostas

### 1. Qual estratégia deu o melhor resultado? Por quê?

**Resposta:** O melhor resultado foi obtido em um empate técnico entre a **Imputação por Constante (0)** e o **Escalonamento com MinMaxScaler**, ambos alcançando a acurácia máxima de **0.7821**.

**Por quê?**
* **MinMaxScaler:** A Regressão Logística calcula seus coeficientes com base na magnitude numérica dos dados. O `MinMaxScaler` comprime as colunas `Age` (idades) e `Fare` (tarifas) para o mesmo intervalo fixo entre $0$ e $1$. Como a coluna de tarifas possui valores muito discrepantes (outliers altos), essa normalização impede que o preço da passagem domine artificialmente os pesos do modelo, equilibrando o aprendizado.
* **Imputação por Constante (0):** Especificamente neste conjunto de dados, preencher as idades faltantes com zero gerou um ganho sutil. Isso acontece porque, no Titanic, a ausência do registro de idade está correlacionada com passageiros da terceira classe (que infelizmente tiveram menor taxa de sobrevivência). Ao aplicar o valor $0$, o modelo linear conseguiu usar essa "anomalia" como um forte indicador preditivo direto.

### 2. O que acontece se você não escalonar os dados?

**Resposta:** Ao desativar o escalonamento numérico, a acurácia foi de **0.7765**, mantendo-se exatamente idêntica ao pipeline padrão que utilizava o `StandardScaler`.

**Por quê?**
Para este conjunto específico de teste do Titanic, o otimizador da Regressão Logística conseguiu convergir para uma solução matemática estável mesmo com as escalas originais.

### 3. O que acontece se você não tratar os valores nulos?

**Resposta:** Ao deixar de aplicar técnicas de imputação e simplesmente adotar a eliminação de linhas com dados faltantes (`dropna`), a acurácia despencou para **0.7622**.

**Por quê?**
A exclusão de registros introduz dois problemas graves na modelagem:
1. **Perda de Informação (Redução Amostral):** Jogamos fora linhas inteiras de dados perfeitamente válidos sobre sexo, classe e tarifa apenas porque a informação da idade não constava, reduzindo a quantidade de dados disponíveis para o modelo aprender.
2. **Viés de Seleção:** Os dados nulos no Titanic não estão distribuídos de forma aleatória. A falta de dados de idade é um padrão concentrado em determinados perfis de passageiros. Deletar essas linhas faz com que o modelo ignore características cruciais dessa população, tornando-se menos robusto e enviesado.

### 4. Qual modelo performou melhor?

**Resposta:** A **Regressão Logística** superou a Árvore de Decisão por uma margem expressiva. Enquanto a Regressão Logística atingiu **0.7821**, o modelo de **Decision Tree** obteve o pior resultado do laboratório, caindo para **0.7263**.

**Por quê?**
Uma Árvore de Decisão treinada com seus parâmetros padrão (sem limite de profundidade) tende a sofrer de **overfitting** (sobreajuste). Ela cria regras tão específicas que decora o conjunto de treino perfeitamente, mas falha ao tentar generalizar os padrões para dados inéditos do conjunto de teste. Por outro lado, a Regressão Logística é um modelo linear mais simples e rígido que atua como um excelente regularizador natural, performando muito melhor em datasets menores e tabulares.